Niniejszy plik służy do pobrania z API Sejmu (https://api.sejm.gov.pl) potrzebnych plików dotyczących interesujących kadencji. Pobrane będą stenogramy posiedzień sejmowych oraz informacje o posłach z danej kadencji. 

In [33]:
import os
import requests
import pickle
import pandas as pd

Wybieram interesujące kadencje: 2015-2019, 2019-2023, 2023-teraz.

In [ ]:
#kadencje
terms = [8, 9, 10]
#folder na stenogramy
OUTPUT_DIR = "stenograms"
os.makedirs(OUTPUT_DIR, exist_ok = True)
for term in terms:
    term_dir = os.path.join(OUTPUT_DIR, str(term))
    os.makedirs(term_dir, exist_ok=True)


Funkcja pobierająca pojedynczy stenogram oraz funkcja pobierająca listę posiedzeń z danej kadencji sejmowej. W trakcie jednej kadencji Sejmu odbywa się wiele posiedzeń, jedno 
posiedzenie może trwać kilka dni - dla każdego dnia z danego posiedzenia jest osobny plik pdf.

In [ ]:
#funkcja pobierania pojedynczego stenogramu
def download_stenogram(session, term, meeting_number, date):
    url = f"https://api.sejm.gov.pl/sejm/term{term}/proceedings/{meeting_number}/{date}/transcripts/pdf"
    r = session.get(url)
    print(f"Pobieram stenograf kadencja {term}, posiedzenie {meeting_number}, data {date}")
    if r.status_code == 200:
        filename = f"posiedzenie_{meeting_number}_{date}.pdf"
        path = os.path.join(OUTPUT_DIR+f"/{term}", filename)
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"Sukces")
    else:
        print(f"Błąd")
#funkcja pobierania listy posiedzień dla danej kadencji 
def download_meetings_list(session, term):
    url = f"https://api.sejm.gov.pl/sejm/term{term}/proceedings"
    r = session.get(url, headers ={"Accept" : "application/json"})
    print(f"Pobieram listę posiedzień kadencji {term}")
    r.raise_for_status()
    return r.json()

Tutaj pobieram listę posiedzeń wraz z datami tych posiedzeń dla każdej kadencji.

In [ ]:
session = requests.Session()

'''
meetings = download_meetings_list(session, term=10)
for i in range(len(meetings)):
    print(f"Posiedzenie: ", meetings[i]["number"])
    print(f"Daty: ", meetings[i]["dates"])
'''

terms_meetings = {}
for term in terms:
    terms_meetings[f"{term}"] = download_meetings_list(session, term = term)


'''#sanity check - czy działa prawidłowo
for term in terms:
    print(f"Kadencja {term}")
    for i in range(len(terms_meetings[f"{term}"])):
        print(f"Posiedzenie: ", terms_meetings[f"{term}"][i]["number"])
        print(f"Daty: ", terms_meetings[f"{term}"][i]["dates"])
'''

Pobieram listę posiedzień kadencji 8
Pobieram listę posiedzień kadencji 9
Pobieram listę posiedzień kadencji 10


Mając już listę posiedzeń i daty dla każdej interesującej mnie kadencji z poprzedniej komórki kodu, mogę przystąpić do pobierania stenogramów i zapisywania ich w formacie pliku pdf. Łącznie jest ok. 1.2 GB danych dla 3 kadencji.

In [ ]:
#pobieram stenogramy posiedzień z każdej kadencji do odpowiedniego folderu
session.headers.update({"Accept" : "application/pdf", "User-Agent" : "Mozilla/5.0 (Python script)"})

for term in terms:
    print(f"Rozpoczynam pobieranie z kadencji {term}")

    meetings = terms_meetings[str(term)]

    for meeting in meetings:
        meeting_number = meeting["number"]

        for date in meeting["dates"]:
            download_stenogram(session, term, meeting_number, date)


Rozpoczynam pobieranie z kadencji 8
Pobieram stenograf kadencja 8, posiedzenie 1, data 2015-11-12
Sukces
Pobieram stenograf kadencja 8, posiedzenie 1, data 2015-11-13
Sukces
Pobieram stenograf kadencja 8, posiedzenie 1, data 2015-11-16
Sukces
Pobieram stenograf kadencja 8, posiedzenie 1, data 2015-11-18
Sukces
Pobieram stenograf kadencja 8, posiedzenie 1, data 2015-11-19
Sukces
Pobieram stenograf kadencja 8, posiedzenie 2, data 2015-11-25
Sukces
Pobieram stenograf kadencja 8, posiedzenie 2, data 2015-11-26
Sukces
Pobieram stenograf kadencja 8, posiedzenie 3, data 2015-12-02
Sukces
Pobieram stenograf kadencja 8, posiedzenie 4, data 2015-12-09
Sukces
Pobieram stenograf kadencja 8, posiedzenie 4, data 2015-12-10
Sukces
Pobieram stenograf kadencja 8, posiedzenie 5, data 2015-12-15
Sukces
Pobieram stenograf kadencja 8, posiedzenie 5, data 2015-12-16
Sukces
Pobieram stenograf kadencja 8, posiedzenie 5, data 2015-12-17
Sukces
Pobieram stenograf kadencja 8, posiedzenie 6, data 2015-12-21
Sukce

Funkcja pobierająca dostępne informacje o posłach danej kadencji.  

In [2]:
#funkcja do pobierania informacji o posłach
def download_mp_list(session, term):
    url = f"https://api.sejm.gov.pl/sejm/term{term}/MP"
    r = session.get(url, headers ={"Accept" : "application/json"})
    print(f"Pobieram listę posłów kadencji {term}")
    r.raise_for_status()
    return r.json()

Pobranie informacji o posłach i zapisanie ich w pliku PKL.

In [34]:
OUTPUT_DIR = 'mps'
terms = [8, 9, 10]
session = requests.Session()
mp_list_term = {}
for term in terms:
    mp_list_term[f'{term}'] = download_mp_list(session, term)
    pickle.dump(mp_list_term[f'{term}'], open(f"mps/kadencja_{term}.pkl", "wb"))


Pobieram listę posłów kadencji 8
Pobieram listę posłów kadencji 9
Pobieram listę posłów kadencji 10


Wypisuję informacje o posłach - imię i nazwisko oraz przynależność partyjna - sprawdzenie czy się pobrało prawidłowo. 

In [ ]:
#wypisywanie wszystkich posłów interesujących mnie kadencji
i = 0
for term in terms:
    print(10*'*', 'Posłowie kadencji: ', term, 10*'*')
    for mp in mp_list_term[f'{term}']:
        i+=1
        club = mp.get('club')
        firstLastName = mp.get('firstLastName')
        
        print(firstLastName, club)


********** Posłowie kadencji:  8 **********
Adam Abramowicz PiS
Andrzej Adamczyk PiS
Zbigniew Ajchler PO-KO
Adam Andruszkiewicz niez.
Waldemar Andzel PiS
Piotr Apel Kukiz15
Dorota Arciszewska-Mielewczyk PiS
Jan Krzysztof Ardanowski PiS
Iwona Arent PiS
Bartosz Arłukowicz PO-KO
Paweł Arndt PO-KO
Marek Ast PiS
Urszula Augustyn PO-KO
Joanna Augustynowska PO-KO
Tadeusz Aziewicz PO-KO
Zbigniew Babalski PiS
Piotr Łukasz Babiarz niez.
Piotr Babinetz PiS
Wojciech Bakun Kukiz15
Paweł Bańkowski PO-KO
Ryszard Bartosik PiS
Barbara Bartuś PiS
Mieczysław Kazimierz Baszko PiS
Dariusz Bąk PiS
Paweł Bejda PSL-KP
Włodzimierz Bernacki PiS
Anna Białkowska PO-KO
Jerzy Bielecki PiS
Marek Biernacki PSL-KP
Zbigniew Biernat PiS
Mariusz Błaszczak PiS
Magdalena Błeńska niez.
Jacek Bogucki PiS
Jerzy Borowczak PO-KO
Joanna Borowiak PiS
Agata Borowiec PiS
Elżbieta Zielińska UPR
Bożena Borys-Szopa PiS
Krzysztof Brejza PO-KO
Joachim Brudziński PiS
Józef Brynkus Kukiz15
Barbara Bubula PiS
Wojciech Buczak PiS
Waldemar B